# 04 — Scaling Analysis

How does performance scale with dataset size and number of features?
When do Foundation Models start outperforming GBDTs?

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.registry import load_dataset, get_all_dataset_names

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

test_df = pd.read_csv('../results/aggregated/test_results.csv')

In [ ]:
# Collect dataset sizes
dataset_sizes = {}
for name in get_all_dataset_names():
    try:
        X, y, info = load_dataset(name)
        dataset_sizes[name] = {'n_samples': len(X), 'n_features': X.shape[1]}
    except Exception:
        pass

sizes_df = pd.DataFrame(dataset_sizes).T
sizes_df.index.name = 'dataset'
sizes_df = sizes_df.reset_index()

# Merge with results
merged = test_df.merge(sizes_df, on='dataset')

In [ ]:
# Performance vs dataset size — grouped by model family
model_families = {
    'xgboost': 'GBDT', 'lightgbm': 'GBDT', 'catboost': 'GBDT',
    'ft_transformer': 'DL', 'tabnet': 'DL', 'saint': 'DL', 'tabm': 'DL',
    'mlp': 'DL', 'realmlp': 'DL',
    'tabpfn': 'Foundation',
}
merged['family'] = merged['model'].map(model_families)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Classification
clf = merged[merged['task_type'].isin(['binary', 'multiclass'])]
if not clf.empty and 'accuracy' in clf.columns:
    for family, group in clf.groupby('family'):
        axes[0].scatter(group['n_samples'], group['accuracy'],
                       label=family, s=60, alpha=0.7)
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Number of Samples (log scale)')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Classification: Accuracy vs Dataset Size')
    axes[0].legend()

# Regression
reg = merged[merged['task_type'] == 'regression']
if not reg.empty and 'r2' in reg.columns:
    for family, group in reg.groupby('family'):
        axes[1].scatter(group['n_samples'], group['r2'],
                       label=family, s=60, alpha=0.7)
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Number of Samples (log scale)')
    axes[1].set_ylabel('R²')
    axes[1].set_title('Regression: R² vs Dataset Size')
    axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/scaling_samples.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Performance vs number of features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (task_label, subset, metric) in zip(axes, [
    ('Classification', merged[merged['task_type'].isin(['binary', 'multiclass'])], 'accuracy'),
    ('Regression', merged[merged['task_type'] == 'regression'], 'r2'),
]):
    if subset.empty or metric not in subset.columns:
        continue
    for family, group in subset.groupby('family'):
        ax.scatter(group['n_features'], group[metric],
                  label=family, s=60, alpha=0.7)
    ax.set_xlabel('Number of Features')
    ax.set_ylabel(metric.upper())
    ax.set_title(f'{task_label}: {metric} vs Feature Count')
    ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/scaling_features.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Performance gap: best GBDT vs best DL/Foundation per dataset
for task_type, metric, higher_is_better in [
    ('binary', 'roc_auc', True),
    ('multiclass', 'accuracy', True),
    ('regression', 'rmse', False),
]:
    subset = merged[merged['task_type'] == task_type]
    if subset.empty or metric not in subset.columns:
        continue

    # Best per family per dataset
    agg_fn = 'max' if higher_is_better else 'min'
    best = subset.groupby(['dataset', 'family'])[metric].agg(agg_fn).unstack('family')

    print(f"\n=== {task_type}: Best {metric} per family ===")
    display(best)

    if 'GBDT' in best.columns and 'DL' in best.columns:
        gap = best['DL'] - best['GBDT'] if higher_is_better else best['GBDT'] - best['DL']
        print(f"\nDL advantage (positive = DL wins):")
        print(gap.sort_values(ascending=False))